# SOTA LLM Embedding

**Goal:** load a pretrained sentence embedding model, generate dense vectors for a word and a sentence, and inspect output dimensions.

**Flow:** Import → Load model (with fallback) → Encode word → Check shape → Encode sentence → Compare dimensions

## Step 1: Import the embedding library

- `SentenceTransformer` provides access to pretrained Hugging Face embedding models.
- Warning filter suppresses noisy `IProgress not found` messages in notebooks.

**Alternative:** use the `transformers` library directly with manual tokenization + pooling for fine-grained control.

In [2]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

from sentence_transformers import SentenceTransformer

print('All imports successful')

All imports successful


## Step 2: Load a Hugging Face embedding model

- Three candidate models are tried in order until one loads.
- `all-MiniLM-L6-v2` is preferred — small, fast, 384-dim output.
- If all fail, a clear `RuntimeError` is raised with collected error messages.

**Why a fallback loop?** Network restrictions or model availability issues shouldn't crash the notebook. The loop makes it robust across different environments.

**Alternative:** download a model locally and pass the folder path to `SentenceTransformer(...)` to avoid network dependency entirely.

In [3]:
model_candidates = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/paraphrase-MiniLM-L6-v2',
    'BAAI/bge-small-en-v1.5',
]

open_source_embedding_model = None
selected_model_name = None
load_errors = {}

for model_name in model_candidates:
    try:
        open_source_embedding_model = SentenceTransformer(model_name)
        selected_model_name = model_name
        break
    except Exception as exc:
        load_errors[model_name] = str(exc)

if open_source_embedding_model is None:
    raise RuntimeError(
        'No embedding model could be loaded. If SSL blocks Hugging Face access, '
        'download one model manually and pass its local folder path to SentenceTransformer(...).\n'
        f'Errors: {load_errors}'
    )

print(f'Embedding model loaded successfully: {selected_model_name}')

Embedding model loaded successfully: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Encode a single word

- `.encode(text)` returns a fixed-size dense NumPy array.
- Even though the model is optimized for sentences, it can still encode single words.
- We check the first few values, shape, and length to confirm the output dimension.

**Why inspect shape?** Different models produce different dimensions (384 for MiniLM, 768 for others). Knowing this is essential before using embeddings in downstream tasks.

In [4]:
text = 'laptop'

In [5]:
embedding = open_source_embedding_model.encode(text)
embedding[:10]

array([-0.05419768,  0.07089285,  0.00296474,  0.00037449,  0.00631793,
       -0.00375075,  0.05877814,  0.03896432,  0.09810222, -0.00604498],
      dtype=float32)

In [6]:
embedding.shape

(384,)

In [7]:
len(embedding)

384

## Step 4: Encode a full sentence

- Sentence transformers are designed for sentence-level input — this is their sweet spot.
- The output has the same dimension as the word embedding (same model, same space).
- The semantic content is different — the model captures meaning, not just characters.

**Key insight:** `"laptop"` and `"This is a test sentence for embedding."` both map to 384-dim vectors, but their positions in embedding space reflect their different meanings.

In [8]:
sentence = "This is a test sentence for embedding."

In [9]:
embedding_sent = open_source_embedding_model.encode(sentence)
embedding_sent[:10]

array([ 0.02782414,  0.00170256,  0.08005551,  0.04666282,  0.038522  ,
        0.05190136,  0.00691998, -0.05277497,  0.01878226, -0.026011  ],
      dtype=float32)

In [10]:
embedding_sent.shape


(384,)

In [11]:
len(embedding_sent)

384

## Revision notes

- `SentenceTransformer` wraps a transformer encoder + pooling into one `.encode()` call.
- Output is a fixed-size dense NumPy array — same dimension for any input length.
- The model tokenizes internally, passes through transformer layers, then mean-pools into one vector.
- Longer inputs get truncated at the model's max sequence length (typically 256–512 tokens).
- Fallback model lists make the notebook robust to network or availability issues.
- These embeddings capture **semantic meaning** — unlike BoW/TF-IDF which are purely lexical.

**Next steps:** use embeddings for similarity search, clustering, or as features for classifiers.

1. **"How are sentence embeddings different from Word2Vec?"**
   Word2Vec gives one static vector per word. Sentence transformers give one contextual vector per input — they capture the meaning of the whole sentence, not individual words.

2. **"What happens if the input exceeds the model's max token length?"**
   It gets truncated. Tokens beyond the limit are silently dropped. For long documents, chunk the text and embed each chunk separately.

3. **"Why does the same model give the same dimension for a word and a sentence?"**
   The model always pools all token representations into a single fixed-size vector (mean pooling by default). Input length affects token count, not output dimension.

4. **"Can you fine-tune these models on your own data?"**
   Yes. `sentence-transformers` supports fine-tuning with contrastive loss, triplet loss, or cosine similarity loss. This improves domain-specific embeddings significantly.

5. **"Why use `SentenceTransformer` instead of raw `transformers`?"**
   `SentenceTransformer` handles tokenization, forward pass, and pooling in one call. With raw `transformers`, you'd write 10+ lines to get the same result.

6. **"Are these embeddings good for keyword search?"**
   No. They're designed for semantic similarity. `"laptop"` and `"notebook computer"` would be close in embedding space, but keyword search would miss the match entirely.

7. **"What's mean pooling and why is it the default?"**
   It averages all token embeddings into one vector. Alternatives include CLS-token pooling or max pooling. Mean pooling works best empirically for most sentence-transformers models.

1. **"Two sentences mean the same thing but use completely different words. How do embeddings handle this?"**
   Well — that's the whole point. `"The car is fast"` and `"The automobile has high speed"` will be close in embedding space because the model learned semantic similarity during pretraining.

2. **"If embeddings capture meaning, can they be wrong?"**
   Yes. Models have biases from training data. Sarcasm, negation (`"not good"` vs `"good"`), and domain-specific jargon can produce misleading embeddings without fine-tuning.

3. **"How would you compare embeddings from two different models?"**
   You can't directly — different models have different vector spaces and dimensions. To compare, you'd need to project into a shared space or compare task performance (e.g., retrieval accuracy).

4. **"Can sentence embeddings replace TF-IDF entirely?"**
   For semantic tasks (search, similarity) — usually yes. For interpretable feature extraction or lightweight baselines — TF-IDF is still useful and much faster.

5. **"What's the computational cost of generating embeddings vs TF-IDF?"**
   TF-IDF is a simple matrix operation — milliseconds. Sentence transformer inference requires a GPU-optimized forward pass — orders of magnitude slower, but captures meaning that TF-IDF cannot.